# Smallest/Largest Node Coverage Plot

This notebook compares node-level CI summaries for the `smallest` and `largest` spillover nodes, plotting both oracle and fitted results on one figure.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 220,
        "font.size": 18,
        "axes.labelsize": 22,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
        "axes.edgecolor": "0.8",
        "axes.linewidth": 0.8,
        "axes.spines.top": True,
        "axes.spines.right": True,
        "grid.alpha": 0.35,
    }
)

# Files to compare
ORACLE_PATH = Path("dataset/metrics_SBM_n3000_K3_sbm_separate_self_nuis-oracle_B200_a0p05_mij_2.json")
FITTED_PATH = Path("dataset/metrics_SBM_n3000_K3_sbm_separate_self_nuis-fitted_B200_a0p05_mij_1.json")

# These line references match alpha_0p1/default/test/individual_spillover in the attached snippet.
# Change ALPHA_KEY to "alpha_0p05" if you want the other alpha branch.
ALPHA_KEY = "alpha_0p1"
K_KEY = "default"
SPLIT = "test"
TARGET = "individual_spillover"
KEEP_LABELS = {"p10", "largest"}


def extract_node_snapshot_rows(path: Path, source: str):
    with path.open("r") as f:
        m = json.load(f)

    target_block = (
        m.get("coverage_by_alpha_k", {})
         .get(ALPHA_KEY, {})
         .get(K_KEY, {})
         .get(SPLIT, {})
         .get(TARGET, {})
    )
    if not target_block:
        raise KeyError(f"Missing block for {ALPHA_KEY}/{K_KEY}/{SPLIT}/{TARGET} in {path}")

    by_prefix = target_block.get("by_prefix", {})
    if not by_prefix:
        raise KeyError(f"No by_prefix found in {path}")

    b_total = target_block.get("B_total")
    key = str(b_total) if (b_total is not None and str(b_total) in by_prefix) else str(max(int(k) for k in by_prefix))

    snapshot_nodes = by_prefix[key].get("node_ci_snapshot", {}).get("nodes", [])
    if not snapshot_nodes:
        raise KeyError(f"No node_ci_snapshot nodes found at prefix {key} in {path}")

    rows = []
    for node in snapshot_nodes:
        label = str(node.get("label", "")).lower()
        if label not in KEEP_LABELS:
            continue
        rows.append({
            "source": source,
            "label": label,
            "node_index": int(node["node_index"]),
            "true_spillover": float(node["true_spillover"]),
            "ci_type": "pointwise",
            "lower": float(node["pointwise_ci_lower"]),
            "upper": float(node["pointwise_ci_upper"]),
        })
        rows.append({
            "source": source,
            "label": label,
            "node_index": int(node["node_index"]),
            "true_spillover": float(node["true_spillover"]),
            "ci_type": "uniform",
            "lower": float(node["uniform_ci_lower"]),
            "upper": float(node["uniform_ci_upper"]),
        })
    return rows


rows = []
rows.extend(extract_node_snapshot_rows(ORACLE_PATH, "oracle"))
rows.extend(extract_node_snapshot_rows(FITTED_PATH, "fitted"))
df = pd.DataFrame(rows)

# Rename labels for presentation/plotting while preserving extraction keys.
df["label"] = df["label"].replace({"p10": "smallest"})
df["center"] = 0.5 * (df["lower"] + df["upper"])
display(df.sort_values(["label", "source", "ci_type"]))

# Shared y layout: one row per (source, ci_type) pair so each row contains
# both smallest and largest intervals.
# Keep oracle rows close together and fitted rows close together, with larger
# separation between oracle and fitted groups.
row_y = {
    ("oracle", "pointwise"): 3.2,
    ("oracle", "uniform"): 2.8,
    ("fitted", "pointwise"): 1.6,
    ("fitted", "uniform"): 1.2,
}

# Requested style:
# - True spillover lines use one color per node label: red=smallest, blue=largest.
# - CI color uses one color per source: gold=oracle, navy=fitted.
# - Pointwise vs uniform uses linestyle.
# - Add visible end caps for CI bars.
true_color = {"smallest": "red", "largest": "blue"}
ci_color = {"oracle": "gold", "fitted": "navy"}
linestyle = {"pointwise": "-", "uniform": "--"}

plot_df = df.copy()
needed_rows = set(zip(plot_df["source"], plot_df["ci_type"]))
missing_rows = needed_rows - set(row_y)
if missing_rows:
    raise KeyError(f"row_y is missing mappings for: {sorted(missing_rows)}")

plot_df["y"] = plot_df.apply(lambda r: row_y[(r["source"], r["ci_type"])], axis=1)

fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")
ax.set_facecolor("white")

cap_half_height = 0.055
for _, r in plot_df.iterrows():
    c = ci_color[r["source"]]
    y = r["y"]

    # CI band line (solid for pointwise, dashed for uniform)
    ax.hlines(y, r["lower"], r["upper"], color=c, linewidth=2.4, linestyle=linestyle[r["ci_type"]])

    # End caps (left/right) to mimic classic error-bar look
    ax.vlines([r["lower"], r["upper"]], y - cap_half_height, y + cap_half_height, color=c, linewidth=1.8)

    # CI midpoint as round dot
    ax.plot(r["center"], y, marker="o", color=c, markersize=6)

# True spillover as full-height vertical reference lines by node label.
for label, g in plot_df.groupby("label", as_index=False):
    true_x = float(g["true_spillover"].iloc[0])
    ax.axvline(true_x, color=true_color[label], linewidth=2.0, linestyle=":")

oracle_center_y = 0.5 * (row_y[("oracle", "pointwise")] + row_y[("oracle", "uniform")])
fitted_center_y = 0.5 * (row_y[("fitted", "pointwise")] + row_y[("fitted", "uniform")])
ax.set_yticks([oracle_center_y, fitted_center_y])
ax.set_yticklabels(["Oracle", "Fitted"])
ax.set_xlabel("Individual spillover value")
ax.set_title("Node CI Comparison: Smallest/Largest by CI Type")
ax.grid(axis="x", linestyle=":", alpha=0.5)
ax.grid(axis="y", alpha=0.0)

legend_handles = [
    Line2D([0], [0], color="gold", lw=2.4, label="CI: Oracle"),
    Line2D([0], [0], color="navy", lw=2.4, label="CI: Estimated (fitted)"),
    Line2D([0], [0], color="black", lw=2.2, linestyle="-", label="Pointwise CI"),
    Line2D([0], [0], color="black", lw=2.2, linestyle="--", label="Uniform CI"),
    Line2D([0], [0], color="red", lw=2.0, linestyle=":", label="True ISE (smallest)"),
    Line2D([0], [0], color="blue", lw=2.0, linestyle=":", label="True ISE (largest)"),
]
ax.legend(handles=legend_handles, loc="best", frameon=True)
plt.tight_layout()
plt.show()
